In [ ]:
# ============================================================
# COMPLETE KAGGLE CODE FOR CMC-81633 REVISION
# UAV Illumination-Aware Multi-Task Color Perception
# Includes:
# 1. SHIFT/OOD main model
# 2. Color-only ablation
# 3. Multi-task without uncertainty weighting
# 4. Bucket-classifier ablation
# 5. LOIO with RUN_LOIO = True
# 6. Policy table, coverage-risk table, failure tables
# 7. ZIP output for download
# ============================================================

import os
import gc
import math
import time
import random
import shutil
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, roc_auc_score

from tqdm.auto import tqdm

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:
    import timm
except Exception:
    !pip -q install timm
    import timm

try:
    import imagehash
except Exception:
    !pip -q install imagehash
    import imagehash

import torchvision.transforms as T

torch.backends.cudnn.benchmark = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


# ============================================================
# 0. Reproducibility
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def gpu_reset():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# ============================================================
# 1. Configuration
# ============================================================

@dataclass
class CFG:
    data_root: str = "/kaggle/input/different-colors-in-challenging-lightening-v2/colorDataset_ml224"

    img_size: int = 224
    model_name: str = "tf_efficientnetv2_s"

    epochs: int = 18
    batch_size: int = 24
    accum_steps: int = 6

    lr: float = 3e-4
    weight_decay: float = 1e-4

    amp: bool = True
    grad_clip: float = 1.0

    label_smoothing: float = 0.05
    drop_rate: float = 0.1
    drop_path_rate: float = 0.1

    n_folds: int = 5
    phash_size: int = 8

    use_bucket_balanced_sampler: bool = True
    use_tta: bool = False

    out_dir: str = "/kaggle/working"
    res_dir: str = "/kaggle/working/results"
    fig_dir: str = "/kaggle/working/figures"
    phash_cache_csv: str = "/kaggle/working/phash_cache.csv"

    # Main controls
    RUN_IN_DOMAIN_5FOLD: bool = False   # change to True only if you want full 5-fold rerun
    RUN_SHIFT_OOD: bool = True

    # Reviewer ablations
    RUN_FAST_ABLATIONS: bool = True
    RUN_BUCKET_ABLATION: bool = True
    RUN_LOIO: bool = True               # requested: TRUE

cfg = CFG()

Path(cfg.res_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.fig_dir).mkdir(parents=True, exist_ok=True)

COLORS = ["Black", "Blue", "Gray", "Orange", "Pink", "Purple", "Skyblue", "White", "Yellow"]
ILLUMS = ["fluorescentLight", "indoor", "indoorNight", "sunLight"]

color2idx = {c: i for i, c in enumerate(COLORS)}
illum2idx = {k: i for i, k in enumerate(ILLUMS)}

print(cfg)


# ============================================================
# 2. Build dataframe
# ============================================================

def build_df(root):
    root = Path(root)
    rows = []
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

    for color in COLORS:
        for illum in ILLUMS:
            folder = root / color / illum
            if not folder.exists():
                print("Missing folder:", folder)
                continue

            for f in folder.iterdir():
                if f.suffix.lower() in exts:
                    rows.append({
                        "path": str(f),
                        "color": color,
                        "illum": illum,
                        "color_id": color2idx[color],
                        "illum_id": illum2idx[illum],
                        "bucket": f"{color}|{illum}"
                    })

    return pd.DataFrame(rows)

df = build_df(cfg.data_root)
print("Dataset shape:", df.shape)
display(df.head())

df["color"].value_counts().reindex(COLORS).to_csv(f"{cfg.res_dir}/dist_color.csv")
df["illum"].value_counts().reindex(ILLUMS).to_csv(f"{cfg.res_dir}/dist_illum.csv")
df["bucket"].value_counts().to_csv(f"{cfg.res_dir}/dist_bucket.csv")


# ============================================================
# 3. pHash grouping
# ============================================================

def compute_phash(path, hash_size=8):
    img = Image.open(path).convert("RGB")
    return str(imagehash.phash(img, hash_size=hash_size))

def add_phash(df0):
    cache_path = Path(cfg.phash_cache_csv)

    if cache_path.exists():
        cached = pd.read_csv(cache_path)
        if set(["path", "phash"]).issubset(cached.columns) and len(cached) == len(df0):
            print("Loaded pHash cache:", cache_path)
            merged = df0.merge(cached[["path", "phash"]], on="path", how="left")
            if merged["phash"].notna().all():
                return merged

    print("Computing pHash cache...")
    phashes = []
    for p in tqdm(df0["path"].values, total=len(df0)):
        phashes.append(compute_phash(p, cfg.phash_size))

    df1 = df0.copy()
    df1["phash"] = phashes
    df1[["path", "phash"]].to_csv(cache_path, index=False)
    print("Saved pHash cache:", cache_path)
    return df1

df = add_phash(df)

bucket2id = {b: i for i, b in enumerate(sorted(df["bucket"].unique()))}
id2bucket = {v: k for k, v in bucket2id.items()}
df["bucket_id"] = df["bucket"].map(bucket2id)

phash_stats = pd.DataFrame({
    "metric": [
        "usable_records",
        "unique_phash_groups",
        "duplicate_affected_records",
        "max_group_size"
    ],
    "value": [
        len(df),
        df["phash"].nunique(),
        int(df["phash"].duplicated(keep=False).sum()),
        int(df["phash"].value_counts().max())
    ]
})

phash_stats.to_csv(f"{cfg.res_dir}/phash_grouping_stats.csv", index=False)
display(phash_stats)


# ============================================================
# 4. Transforms
# ============================================================

train_tfms = T.Compose([
    T.RandomResizedCrop(cfg.img_size, scale=(0.75, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.04, hue=0.0),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

valid_tfms = T.Compose([
    T.Resize(int(cfg.img_size * 1.14)),
    T.CenterCrop(cfg.img_size),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])


# ============================================================
# 5. Dataset and loaders
# ============================================================

class ColorIllumDataset(Dataset):
    def __init__(self, df_, tfms=None):
        self.df = df_.reset_index(drop=True)
        self.tfms = tfms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(r["path"]).convert("RGB")
        if self.tfms:
            img = self.tfms(img)

        return {
            "image": img,
            "color": torch.tensor(r["color_id"], dtype=torch.long),
            "illum": torch.tensor(r["illum_id"], dtype=torch.long),
            "bucket_id": torch.tensor(r["bucket_id"], dtype=torch.long),
            "bucket": r["bucket"],
            "path": r["path"]
        }

def collate_fn(batch):
    images = torch.stack([b["image"] for b in batch])
    colors = torch.stack([b["color"] for b in batch])
    illums = torch.stack([b["illum"] for b in batch])
    bucket_ids = torch.stack([b["bucket_id"] for b in batch])
    buckets = [b["bucket"] for b in batch]
    paths = [b["path"] for b in batch]
    return images, colors, illums, bucket_ids, buckets, paths

def make_bucket_sampler(df_sub):
    counts = df_sub["bucket"].value_counts().to_dict()
    weights = df_sub["bucket"].map(lambda x: 1.0 / counts[x]).values.astype(np.float32)

    return WeightedRandomSampler(
        torch.tensor(weights, dtype=torch.double),
        num_samples=len(weights),
        replacement=True
    )

def make_loader(df_sub, tfms, shuffle=False, sampler=None, batch_size=None):
    if batch_size is None:
        batch_size = cfg.batch_size

    ds = ColorIllumDataset(df_sub, tfms=tfms)

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=(sampler is None and shuffle),
        sampler=sampler,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
        drop_last=False,
        collate_fn=collate_fn
    )


# ============================================================
# 6. Split functions
# ============================================================

def make_group_stratified_fold(df_all, fold=0, n_splits=5, strat_col="bucket"):
    y = df_all[strat_col].values
    groups = df_all["phash"].values
    idx = np.arange(len(df_all))

    skf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for i, (tr, va) in enumerate(skf.split(idx, y=y, groups=groups)):
        if i == fold:
            return df_all.iloc[tr].reset_index(drop=True), df_all.iloc[va].reset_index(drop=True)

    raise ValueError("Fold not found")

def make_shift_split(df_all):
    train_ills = {"indoor", "fluorescentLight"}
    test_ills = {"sunLight", "indoorNight"}

    df_train = df_all[df_all["illum"].isin(train_ills)].reset_index(drop=True)
    df_test = df_all[df_all["illum"].isin(test_ills)].reset_index(drop=True)

    return df_train, df_test


# ============================================================
# 7. Models
# ============================================================

class Backbone(nn.Module):
    def __init__(self, model_name, drop_rate=0.1, drop_path_rate=0.1):
        super().__init__()
        self.net = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,
            global_pool="avg",
            drop_rate=drop_rate,
            drop_path_rate=drop_path_rate
        )
        self.feat_dim = self.net.num_features

    def forward(self, x):
        return self.net(x)

class MultiTaskNet(nn.Module):
    def __init__(self, model_name, n_color=9, n_illum=4, drop_rate=0.1, drop_path_rate=0.1, uncertainty=True):
        super().__init__()
        self.backbone = Backbone(model_name, drop_rate, drop_path_rate)
        self.color_head = nn.Linear(self.backbone.feat_dim, n_color)
        self.illum_head = nn.Linear(self.backbone.feat_dim, n_illum)
        self.uncertainty = uncertainty

        if uncertainty:
            self.log_var_color = nn.Parameter(torch.zeros(()))
            self.log_var_illum = nn.Parameter(torch.zeros(()))

    def forward(self, x):
        f = self.backbone(x)
        return self.color_head(f), self.illum_head(f)

class ColorOnlyNet(nn.Module):
    def __init__(self, model_name, n_color=9, drop_rate=0.1, drop_path_rate=0.1):
        super().__init__()
        self.backbone = Backbone(model_name, drop_rate, drop_path_rate)
        self.color_head = nn.Linear(self.backbone.feat_dim, n_color)

    def forward(self, x):
        f = self.backbone(x)
        return self.color_head(f)

class BucketNet(nn.Module):
    def __init__(self, model_name, n_bucket=36, drop_rate=0.1, drop_path_rate=0.1):
        super().__init__()
        self.backbone = Backbone(model_name, drop_rate, drop_path_rate)
        self.bucket_head = nn.Linear(self.backbone.feat_dim, n_bucket)

    def forward(self, x):
        f = self.backbone(x)
        return self.bucket_head(f)


# ============================================================
# 8. Loss, optimizer, scheduler
# ============================================================

def soft_ce_with_ls(logits, target, smoothing=0.0):
    n = logits.size(-1)
    logp = F.log_softmax(logits, dim=-1)

    with torch.no_grad():
        true_dist = torch.zeros_like(logp)
        true_dist.fill_(smoothing / (n - 1))
        true_dist.scatter_(1, target.unsqueeze(1), 1.0 - smoothing)

    return torch.mean(torch.sum(-true_dist * logp, dim=-1))

def make_optim(model):
    return torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

def make_scheduler(optimizer, total_steps):
    warmup = int(0.06 * total_steps)

    def lr_lambda(step):
        if step < warmup:
            return float(step) / float(max(1, warmup))

        progress = (step - warmup) / float(max(1, total_steps - warmup))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ============================================================
# 9. Metrics
# ============================================================

def compute_ece(probs, y_true, n_bins=15):
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    acc = (pred == y_true).astype(np.float32)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for b0, b1 in zip(bins[:-1], bins[1:]):
        mask = (conf > b0) & (conf <= b1)
        if mask.sum() == 0:
            continue
        ece += np.abs(acc[mask].mean() - conf[mask].mean()) * mask.mean()

    return float(ece)

def entropy_from_probs(probs):
    return -(probs * np.log(np.maximum(probs, 1e-12))).sum(axis=1)

def entropy_from_logits(logits):
    p = np.exp(logits - logits.max(axis=1, keepdims=True))
    p = p / np.maximum(p.sum(axis=1, keepdims=True), 1e-12)
    return entropy_from_probs(p)

def plot_confusion(cm, class_names, title, save_path):
    cm_plot = cm.astype(np.float32)
    cm_plot = cm_plot / np.maximum(cm_plot.sum(axis=1, keepdims=True), 1e-12)

    fig = plt.figure(figsize=(8, 6))
    plt.imshow(cm_plot, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=45, ha="right")
    plt.yticks(ticks, class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    fig.savefig(save_path, dpi=300)
    plt.close(fig)


# ============================================================
# 10. Evaluation
# ============================================================

@torch.no_grad()
def predict_multitask(model, x):
    with torch.amp.autocast(device_type="cuda", enabled=cfg.amp and torch.cuda.is_available()):
        lc, li = model(x)
    return lc, li

@torch.no_grad()
def eval_multitask(model, loader, save_log_path=None, protocol=""):
    model.eval()

    y_color_all, y_illum_all, y_bucket_all = [], [], []
    pred_color_all, pred_illum_all, pred_bucket_all = [], [], []
    probs_color_all = []

    rows = []
    sample_id = 0

    for x, yc, yi, bid, buckets, paths in loader:
        x = x.to(device, non_blocking=True).contiguous(memory_format=torch.channels_last)

        lc, li = predict_multitask(model, x)

        pc = F.softmax(lc, dim=1).detach().cpu().numpy()
        pi = F.softmax(li, dim=1).detach().cpu().numpy()

        pred_c = pc.argmax(axis=1)
        pred_i = pi.argmax(axis=1)

        yc_np = yc.numpy()
        yi_np = yi.numpy()
        bid_np = bid.numpy()

        pred_b = np.array([
            bucket2id[f"{COLORS[c]}|{ILLUMS[i]}"]
            for c, i in zip(pred_c, pred_i)
        ])

        y_color_all.append(yc_np)
        y_illum_all.append(yi_np)
        y_bucket_all.append(bid_np)

        pred_color_all.append(pred_c)
        pred_illum_all.append(pred_i)
        pred_bucket_all.append(pred_b)
        probs_color_all.append(pc)

        if save_log_path is not None:
            pmax = pc.max(axis=1)
            entropy = entropy_from_probs(pc)
            illum_pmax = pi.max(axis=1)

            for j in range(len(yc_np)):
                rows.append({
                    "sample_id": sample_id,
                    "path": paths[j],
                    "true_color_id": int(yc_np[j]),
                    "pred_color_id": int(pred_c[j]),
                    "true_color": COLORS[int(yc_np[j])],
                    "pred_color": COLORS[int(pred_c[j])],
                    "true_illum_id": int(yi_np[j]),
                    "pred_illum_id": int(pred_i[j]),
                    "true_illum": ILLUMS[int(yi_np[j])],
                    "pred_illum": ILLUMS[int(pred_i[j])],
                    "bucket": buckets[j],
                    "pmax": float(pmax[j]),
                    "entropy": float(entropy[j]),
                    "illum_pmax": float(illum_pmax[j]),
                    "correct": int(pred_c[j] == yc_np[j]),
                    "protocol": protocol
                })
                sample_id += 1

    y_color = np.concatenate(y_color_all)
    y_illum = np.concatenate(y_illum_all)
    y_bucket = np.concatenate(y_bucket_all)

    pred_color = np.concatenate(pred_color_all)
    pred_illum = np.concatenate(pred_illum_all)
    pred_bucket = np.concatenate(pred_bucket_all)

    probs_color = np.concatenate(probs_color_all)

    if save_log_path is not None:
        pd.DataFrame(rows).to_csv(save_log_path, index=False)

    return {
        "color_macro_f1": float(f1_score(y_color, pred_color, average="macro")),
        "illum_macro_f1": float(f1_score(y_illum, pred_illum, average="macro")),
        "bucket_macro_f1": float(f1_score(y_bucket, pred_bucket, average="macro")),
        "ece": float(compute_ece(probs_color, y_color)),
        "cm_color": confusion_matrix(y_color, pred_color, labels=list(range(len(COLORS)))),
        "cm_illum": confusion_matrix(y_illum, pred_illum, labels=list(range(len(ILLUMS))))
    }

@torch.no_grad()
def eval_shift_ood(model, loader_seen, loader_unseen, save_log_path=None):
    model.eval()

    illum_logits_seen = []
    illum_logits_unseen = []

    for x, yc, yi, bid, buckets, paths in loader_seen:
        x = x.to(device, non_blocking=True).contiguous(memory_format=torch.channels_last)
        lc, li = predict_multitask(model, x)
        illum_logits_seen.append(li.detach().cpu().numpy())

    unseen_metrics = eval_multitask(
        model,
        loader_unseen,
        save_log_path=save_log_path,
        protocol="SHIFT_OOD"
    )

    for x, yc, yi, bid, buckets, paths in loader_unseen:
        x = x.to(device, non_blocking=True).contiguous(memory_format=torch.channels_last)
        lc, li = predict_multitask(model, x)
        illum_logits_unseen.append(li.detach().cpu().numpy())

    score_seen = entropy_from_logits(np.concatenate(illum_logits_seen))
    score_unseen = entropy_from_logits(np.concatenate(illum_logits_unseen))

    y_ood = np.concatenate([np.zeros_like(score_seen), np.ones_like(score_unseen)])
    scores = np.concatenate([score_seen, score_unseen])

    auroc = roc_auc_score(y_ood, scores)

    return {
        "color_macro_f1_unseen": unseen_metrics["color_macro_f1"],
        "illum_macro_f1_unseen": unseen_metrics["illum_macro_f1"],
        "bucket_macro_f1_unseen": unseen_metrics["bucket_macro_f1"],
        "color_ece_unseen": unseen_metrics["ece"],
        "illum_ood_auroc_entropy": float(auroc),
        "cm_color_unseen": unseen_metrics["cm_color"],
        "cm_illum_unseen": unseen_metrics["cm_illum"]
    }

@torch.no_grad()
def eval_color_only(model, loader, save_log_path=None, protocol=""):
    model.eval()

    y_all, pred_all, probs_all = [], [], []
    rows = []
    sample_id = 0

    for x, yc, yi, bid, buckets, paths in loader:
        x = x.to(device, non_blocking=True).contiguous(memory_format=torch.channels_last)

        with torch.amp.autocast(device_type="cuda", enabled=cfg.amp and torch.cuda.is_available()):
            logits = model(x)

        probs = F.softmax(logits, dim=1).detach().cpu().numpy()
        pred = probs.argmax(axis=1)

        yc_np = yc.numpy()

        y_all.append(yc_np)
        pred_all.append(pred)
        probs_all.append(probs)

        if save_log_path is not None:
            pmax = probs.max(axis=1)
            entropy = entropy_from_probs(probs)

            for j in range(len(yc_np)):
                rows.append({
                    "sample_id": sample_id,
                    "path": paths[j],
                    "true_color_id": int(yc_np[j]),
                    "pred_color_id": int(pred[j]),
                    "true_color": COLORS[int(yc_np[j])],
                    "pred_color": COLORS[int(pred[j])],
                    "pmax": float(pmax[j]),
                    "entropy": float(entropy[j]),
                    "correct": int(pred[j] == yc_np[j]),
                    "protocol": protocol
                })
                sample_id += 1

    y = np.concatenate(y_all)
    p = np.concatenate(pred_all)
    probs = np.concatenate(probs_all)

    if save_log_path is not None:
        pd.DataFrame(rows).to_csv(save_log_path, index=False)

    return {
        "color_macro_f1": float(f1_score(y, p, average="macro")),
        "color_accuracy": float(accuracy_score(y, p)),
        "ece": float(compute_ece(probs, y))
    }

@torch.no_grad()
def eval_bucket_model(model, loader):
    model.eval()

    y_bucket_all, pred_bucket_all = [], []
    y_color_all, y_illum_all = [], []
    pred_color_all, pred_illum_all = [], []

    for x, yc, yi, bid, buckets, paths in loader:
        x = x.to(device, non_blocking=True).contiguous(memory_format=torch.channels_last)

        with torch.amp.autocast(device_type="cuda", enabled=cfg.amp and torch.cuda.is_available()):
            logits = model(x)

        pred_bid = logits.detach().cpu().numpy().argmax(axis=1)

        pred_c, pred_i = [], []
        for b in pred_bid:
            bname = id2bucket[int(b)]
            cname, iname = bname.split("|")
            pred_c.append(color2idx[cname])
            pred_i.append(illum2idx[iname])

        y_bucket_all.append(bid.numpy())
        pred_bucket_all.append(pred_bid)

        y_color_all.append(yc.numpy())
        y_illum_all.append(yi.numpy())

        pred_color_all.append(np.array(pred_c))
        pred_illum_all.append(np.array(pred_i))

    yb = np.concatenate(y_bucket_all)
    pb = np.concatenate(pred_bucket_all)

    yc = np.concatenate(y_color_all)
    yi = np.concatenate(y_illum_all)

    pc = np.concatenate(pred_color_all)
    pi = np.concatenate(pred_illum_all)

    return {
        "color_macro_f1": float(f1_score(yc, pc, average="macro")),
        "illum_macro_f1": float(f1_score(yi, pi, average="macro")),
        "bucket_macro_f1": float(f1_score(yb, pb, average="macro"))
    }


# ============================================================
# 11. Training
# ============================================================

def train_multitask(df_train, df_valid, tag, uncertainty=True):
    gpu_reset()
    seed_everything(42)

    sampler = make_bucket_sampler(df_train) if cfg.use_bucket_balanced_sampler else None
    dl_train = make_loader(df_train, train_tfms, shuffle=True, sampler=sampler)
    dl_valid = make_loader(df_valid, valid_tfms, shuffle=False)

    model = MultiTaskNet(
        cfg.model_name,
        9,
        4,
        cfg.drop_rate,
        cfg.drop_path_rate,
        uncertainty=uncertainty
    ).to(device)

    model = model.to(memory_format=torch.channels_last)

    optimizer = make_optim(model)
    total_steps = (len(dl_train) // cfg.accum_steps + 1) * cfg.epochs
    scheduler = make_scheduler(optimizer, total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.amp and torch.cuda.is_available())

    best_score = -1
    best_state = None
    history = []

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        t0 = time.time()

        optimizer.zero_grad(set_to_none=True)
        total_loss, n_samples = 0.0, 0

        for step, (x, yc, yi, bid, buckets, paths) in enumerate(dl_train):
            x = x.to(device, non_blocking=True).contiguous(memory_format=torch.channels_last)
            yc = yc.to(device, non_blocking=True)
            yi = yi.to(device, non_blocking=True)

            with torch.amp.autocast(device_type="cuda", enabled=cfg.amp and torch.cuda.is_available()):
                lc, li = model(x)

                Lc = soft_ce_with_ls(lc, yc, cfg.label_smoothing)
                Li = soft_ce_with_ls(li, yi, cfg.label_smoothing)

                if uncertainty:
                    loss = torch.exp(-model.log_var_color) * Lc + model.log_var_color
                    loss += torch.exp(-model.log_var_illum) * Li + model.log_var_illum
                else:
                    loss = Lc + Li

                loss = loss / cfg.accum_steps

            scaler.scale(loss).backward()

            if (step + 1) % cfg.accum_steps == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            bs = x.size(0)
            total_loss += loss.item() * bs * cfg.accum_steps
            n_samples += bs

        val = eval_multitask(model, dl_valid)

        if val["bucket_macro_f1"] > best_score:
            best_score = val["bucket_macro_f1"]
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

        row = {
            "epoch": epoch,
            "train_loss": total_loss / max(1, n_samples),
            "val_color_f1": val["color_macro_f1"],
            "val_illum_f1": val["illum_macro_f1"],
            "val_bucket_f1": val["bucket_macro_f1"],
            "val_ece": val["ece"],
            "time_sec": time.time() - t0
        }

        history.append(row)

        print(
            f"[{tag}] ep {epoch:02d}/{cfg.epochs} "
            f"loss={row['train_loss']:.4f} "
            f"colorF1={row['val_color_f1']:.4f} "
            f"illumF1={row['val_illum_f1']:.4f} "
            f"bucketF1={row['val_bucket_f1']:.4f} "
            f"ECE={row['val_ece']:.4f} "
            f"time={row['time_sec']:.1f}s"
        )

    model.load_state_dict(best_state, strict=True)
    pd.DataFrame(history).to_csv(f"{cfg.res_dir}/history_{tag}.csv", index=False)

    final = eval_multitask(model, dl_valid)

    pd.DataFrame([{
        "tag": tag,
        "color_macro_f1": final["color_macro_f1"],
        "illum_macro_f1": final["illum_macro_f1"],
        "bucket_macro_f1": final["bucket_macro_f1"],
        "ece": final["ece"]
    }]).to_csv(f"{cfg.res_dir}/metrics_{tag}.csv", index=False)

    return model, final

def train_color_only(df_train, df_valid, tag):
    gpu_reset()
    seed_everything(42)

    sampler = make_bucket_sampler(df_train) if cfg.use_bucket_balanced_sampler else None
    dl_train = make_loader(df_train, train_tfms, shuffle=True, sampler=sampler)
    dl_valid = make_loader(df_valid, valid_tfms, shuffle=False)

    model = ColorOnlyNet(cfg.model_name, 9, cfg.drop_rate, cfg.drop_path_rate).to(device)
    model = model.to(memory_format=torch.channels_last)

    optimizer = make_optim(model)
    total_steps = (len(dl_train) // cfg.accum_steps + 1) * cfg.epochs
    scheduler = make_scheduler(optimizer, total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.amp and torch.cuda.is_available())

    best_score = -1
    best_state = None
    history = []

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        t0 = time.time()

        optimizer.zero_grad(set_to_none=True)
        total_loss, n_samples = 0.0, 0

        for step, (x, yc, yi, bid, buckets, paths) in enumerate(dl_train):
            x = x.to(device, non_blocking=True).contiguous(memory_format=torch.channels_last)
            yc = yc.to(device, non_blocking=True)

            with torch.amp.autocast(device_type="cuda", enabled=cfg.amp and torch.cuda.is_available()):
                logits = model(x)
                loss = soft_ce_with_ls(logits, yc, cfg.label_smoothing)
                loss = loss / cfg.accum_steps

            scaler.scale(loss).backward()

            if (step + 1) % cfg.accum_steps == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            bs = x.size(0)
            total_loss += loss.item() * bs * cfg.accum_steps
            n_samples += bs

        val = eval_color_only(model, dl_valid)

        if val["color_macro_f1"] > best_score:
            best_score = val["color_macro_f1"]
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

        row = {
            "epoch": epoch,
            "train_loss": total_loss / max(1, n_samples),
            "val_color_f1": val["color_macro_f1"],
            "val_ece": val["ece"],
            "time_sec": time.time() - t0
        }

        history.append(row)

        print(
            f"[{tag}] ep {epoch:02d}/{cfg.epochs} "
            f"loss={row['train_loss']:.4f} "
            f"colorF1={row['val_color_f1']:.4f} "
            f"ECE={row['val_ece']:.4f} "
            f"time={row['time_sec']:.1f}s"
        )

    model.load_state_dict(best_state, strict=True)
    pd.DataFrame(history).to_csv(f"{cfg.res_dir}/history_{tag}.csv", index=False)

    return model

def train_bucket_model(df_train, df_valid, tag):
    gpu_reset()
    seed_everything(42)

    sampler = make_bucket_sampler(df_train) if cfg.use_bucket_balanced_sampler else None
    dl_train = make_loader(df_train, train_tfms, shuffle=True, sampler=sampler)
    dl_valid = make_loader(df_valid, valid_tfms, shuffle=False)

    model = BucketNet(cfg.model_name, len(bucket2id), cfg.drop_rate, cfg.drop_path_rate).to(device)
    model = model.to(memory_format=torch.channels_last)

    optimizer = make_optim(model)
    total_steps = (len(dl_train) // cfg.accum_steps + 1) * cfg.epochs
    scheduler = make_scheduler(optimizer, total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.amp and torch.cuda.is_available())

    best_score = -1
    best_state = None
    history = []

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        t0 = time.time()

        optimizer.zero_grad(set_to_none=True)
        total_loss, n_samples = 0.0, 0

        for step, (x, yc, yi, bid, buckets, paths) in enumerate(dl_train):
            x = x.to(device, non_blocking=True).contiguous(memory_format=torch.channels_last)
            bid = bid.to(device, non_blocking=True)

            with torch.amp.autocast(device_type="cuda", enabled=cfg.amp and torch.cuda.is_available()):
                logits = model(x)
                loss = soft_ce_with_ls(logits, bid, cfg.label_smoothing)
                loss = loss / cfg.accum_steps

            scaler.scale(loss).backward()

            if (step + 1) % cfg.accum_steps == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            bs = x.size(0)
            total_loss += loss.item() * bs * cfg.accum_steps
            n_samples += bs

        val = eval_bucket_model(model, dl_valid)

        if val["bucket_macro_f1"] > best_score:
            best_score = val["bucket_macro_f1"]
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

        row = {
            "epoch": epoch,
            "train_loss": total_loss / max(1, n_samples),
            "val_color_f1": val["color_macro_f1"],
            "val_illum_f1": val["illum_macro_f1"],
            "val_bucket_f1": val["bucket_macro_f1"],
            "time_sec": time.time() - t0
        }

        history.append(row)

        print(
            f"[{tag}] ep {epoch:02d}/{cfg.epochs} "
            f"loss={row['train_loss']:.4f} "
            f"colorF1={row['val_color_f1']:.4f} "
            f"illumF1={row['val_illum_f1']:.4f} "
            f"bucketF1={row['val_bucket_f1']:.4f} "
            f"time={row['time_sec']:.1f}s"
        )

    model.load_state_dict(best_state, strict=True)
    pd.DataFrame(history).to_csv(f"{cfg.res_dir}/history_{tag}.csv", index=False)

    return model


# ============================================================
# 12. Policy analysis
# ============================================================

def run_policy_analysis(pred_csv):
    pred_df = pd.read_csv(pred_csv)
    total = len(pred_df)

    TAU_ACCEPT = 0.60
    TAU_REOBSERVE = 0.40
    ENTROPY_ACCEPT_MAX = 2.10
    ENTROPY_REOBSERVE_HIGH = 2.10

    def assign_action(row):
        if row["pmax"] >= TAU_ACCEPT and row["entropy"] <= ENTROPY_ACCEPT_MAX:
            return "ACCEPT"
        elif row["pmax"] < TAU_REOBSERVE or row["entropy"] > ENTROPY_REOBSERVE_HIGH:
            return "RE_OBSERVE"
        else:
            return "EDGE_COMPUTE"

    pred_df["action"] = pred_df.apply(assign_action, axis=1)
    pred_df["unsafe_acceptance"] = (
        (pred_df["action"] == "ACCEPT") &
        (pred_df["correct"] == 0)
    ).astype(int)

    policy_rows = []

    for action in ["ACCEPT", "RE_OBSERVE", "EDGE_COMPUTE", "SEND_TO_GS"]:
        sub = pred_df[pred_df["action"] == action]
        n = len(sub)

        if n > 0:
            acc = sub["correct"].mean()
            risk = 1.0 - acc
            unsafe = int(sub["unsafe_acceptance"].sum())
            unsafe_rate = unsafe / total
        else:
            acc = np.nan
            risk = np.nan
            unsafe = 0
            unsafe_rate = 0.0

        policy_rows.append({
            "Action": action,
            "Samples": n,
            "Trigger Rate (%)": round(100 * n / total, 2),
            "Accuracy within Action (%)": round(100 * acc, 2) if n > 0 else "--",
            "Selective Risk within Action (%)": round(100 * risk, 2) if n > 0 else "--",
            "Unsafe Acceptances": unsafe,
            "Unsafe Acceptance Rate (%)": round(100 * unsafe_rate, 2)
        })

    policy_table = pd.DataFrame(policy_rows)
    policy_table.to_csv(f"{cfg.res_dir}/policy_table_shift_ood.csv", index=False)
    pred_df.to_csv(f"{cfg.res_dir}/shift_prediction_log_with_actions.csv", index=False)

    thresholds = np.arange(0.30, 0.91, 0.05)
    coverage_rows = []

    for tau in thresholds:
        accepted = pred_df[pred_df["pmax"] >= tau]
        n_acc = len(accepted)

        if n_acc > 0:
            acc = accepted["correct"].mean()
            risk = 1.0 - acc
            unsafe = int((accepted["correct"] == 0).sum())
        else:
            acc = np.nan
            risk = np.nan
            unsafe = 0

        coverage_rows.append({
            "Threshold": round(float(tau), 2),
            "Accepted Samples": n_acc,
            "Coverage (%)": round(100 * n_acc / total, 2),
            "Accepted Accuracy (%)": round(100 * acc, 2) if n_acc > 0 else "--",
            "Selective Risk (%)": round(100 * risk, 2) if n_acc > 0 else "--",
            "Unsafe Accepted Samples": unsafe
        })

    coverage_table = pd.DataFrame(coverage_rows)
    coverage_table.to_csv(f"{cfg.res_dir}/coverage_risk_shift_ood.csv", index=False)

    plot_df = coverage_table[coverage_table["Selective Risk (%)"] != "--"].copy()
    plot_df["Coverage (%)"] = plot_df["Coverage (%)"].astype(float)
    plot_df["Selective Risk (%)"] = plot_df["Selective Risk (%)"].astype(float)

    fig = plt.figure(figsize=(7, 5))
    plt.plot(plot_df["Coverage (%)"], plot_df["Selective Risk (%)"], marker="o")
    plt.xlabel("Coverage (%)")
    plt.ylabel("Selective Risk (%)")
    plt.title("Coverage-Risk Curve under Illumination Shift")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(f"{cfg.fig_dir}/coverage_risk_shift_ood.png", dpi=300)
    plt.close(fig)

    per_color = (
        pred_df.groupby("true_color")
        .agg(
            Samples=("correct", "size"),
            Correct=("correct", "sum"),
            Mean_Confidence=("pmax", "mean"),
            Mean_Entropy=("entropy", "mean"),
            Accept_Count=("action", lambda x: (x == "ACCEPT").sum()),
            Reobserve_Count=("action", lambda x: (x == "RE_OBSERVE").sum()),
            Edge_Count=("action", lambda x: (x == "EDGE_COMPUTE").sum())
        )
        .reset_index()
    )

    per_color["Accuracy (%)"] = round(100 * per_color["Correct"] / per_color["Samples"], 2)
    per_color["Error Rate (%)"] = round(100 - per_color["Accuracy (%)"], 2)
    per_color["Mean_Confidence"] = per_color["Mean_Confidence"].round(4)
    per_color["Mean_Entropy"] = per_color["Mean_Entropy"].round(4)

    per_color = per_color[
        [
            "true_color", "Samples", "Accuracy (%)", "Error Rate (%)",
            "Mean_Confidence", "Mean_Entropy",
            "Accept_Count", "Reobserve_Count", "Edge_Count"
        ]
    ].sort_values("Error Rate (%)", ascending=False)

    per_color.to_csv(f"{cfg.res_dir}/per_color_shift_failure_table.csv", index=False)

    per_illum = (
        pred_df.groupby("true_illum")
        .agg(
            Samples=("correct", "size"),
            Correct=("correct", "sum"),
            Mean_Confidence=("pmax", "mean"),
            Mean_Entropy=("entropy", "mean"),
            Accept_Count=("action", lambda x: (x == "ACCEPT").sum()),
            Reobserve_Count=("action", lambda x: (x == "RE_OBSERVE").sum()),
            Edge_Count=("action", lambda x: (x == "EDGE_COMPUTE").sum())
        )
        .reset_index()
    )

    per_illum["Accuracy (%)"] = round(100 * per_illum["Correct"] / per_illum["Samples"], 2)
    per_illum["Error Rate (%)"] = round(100 - per_illum["Accuracy (%)"], 2)
    per_illum["Mean_Confidence"] = per_illum["Mean_Confidence"].round(4)
    per_illum["Mean_Entropy"] = per_illum["Mean_Entropy"].round(4)

    per_illum = per_illum[
        [
            "true_illum", "Samples", "Accuracy (%)", "Error Rate (%)",
            "Mean_Confidence", "Mean_Entropy",
            "Accept_Count", "Reobserve_Count", "Edge_Count"
        ]
    ].sort_values("Error Rate (%)", ascending=False)

    per_illum.to_csv(f"{cfg.res_dir}/per_illum_shift_failure_table.csv", index=False)

    print("\nPolicy table")
    display(policy_table)

    print("\nCoverage-risk table")
    display(coverage_table)

    return policy_table, coverage_table


# ============================================================
# 13. Main experiments
# ============================================================

df_shift_train_all, df_shift_test_unseen = make_shift_split(df)
df_shift_tr, df_shift_val_seen = make_group_stratified_fold(df_shift_train_all, fold=0, n_splits=5)

# Optional 5-fold rerun
if cfg.RUN_IN_DOMAIN_5FOLD:
    all_rows = []

    print("\n================ IN-DOMAIN 5-FOLD ================")

    for fold in range(cfg.n_folds):
        df_tr, df_va = make_group_stratified_fold(df, fold=fold, n_splits=cfg.n_folds)

        model_fold, mt = train_multitask(
            df_tr,
            df_va,
            tag=f"IN_multitask_fold{fold}",
            uncertainty=True
        )

        all_rows.append({
            "protocol": "IN",
            "model": "MultiTask-uncertainty",
            "fold": fold,
            "color_macro_f1": mt["color_macro_f1"],
            "illum_macro_f1": mt["illum_macro_f1"],
            "bucket_macro_f1": mt["bucket_macro_f1"],
            "ece": mt["ece"]
        })

    df_all = pd.DataFrame(all_rows)
    df_all.to_csv(f"{cfg.res_dir}/cvpr_all_runs.csv", index=False)

    summary = pd.DataFrame([{
        "protocol": "IN(5-fold)",
        "color_macro_f1_mean": df_all["color_macro_f1"].mean(),
        "color_macro_f1_std": df_all["color_macro_f1"].std(ddof=0),
        "illum_macro_f1_mean": df_all["illum_macro_f1"].mean(),
        "illum_macro_f1_std": df_all["illum_macro_f1"].std(ddof=0),
        "bucket_macro_f1_mean": df_all["bucket_macro_f1"].mean(),
        "bucket_macro_f1_std": df_all["bucket_macro_f1"].std(ddof=0),
        "ece_mean": df_all["ece"].mean(),
        "ece_std": df_all["ece"].std(ddof=0)
    }])

    summary.to_csv(f"{cfg.res_dir}/summary_in_domain.csv", index=False)
    display(summary)


# SHIFT/OOD main model
if cfg.RUN_SHIFT_OOD:
    print("\n================ SHIFT/OOD MAIN MODEL ================")

    shift_model, shift_val_metrics = train_multitask(
        df_shift_tr,
        df_shift_val_seen,
        tag="SHIFT_multitask_uncertainty",
        uncertainty=True
    )

    dl_seen = make_loader(df_shift_val_seen, valid_tfms, shuffle=False)
    dl_unseen = make_loader(df_shift_test_unseen, valid_tfms, shuffle=False)

    shift_log_path = f"{cfg.res_dir}/shift_prediction_log.csv"

    shift_metrics = eval_shift_ood(
        shift_model,
        dl_seen,
        dl_unseen,
        save_log_path=shift_log_path
    )

    shift_table = pd.DataFrame([{
        "protocol": "SHIFT_OOD",
        "model": "MultiTask-uncertainty",
        "color_macro_f1_unseen": shift_metrics["color_macro_f1_unseen"],
        "illum_macro_f1_unseen": shift_metrics["illum_macro_f1_unseen"],
        "bucket_macro_f1_unseen": shift_metrics["bucket_macro_f1_unseen"],
        "color_ece_unseen": shift_metrics["color_ece_unseen"],
        "illum_ood_auroc_entropy": shift_metrics["illum_ood_auroc_entropy"]
    }])

    shift_table.to_csv(f"{cfg.res_dir}/shift_ood_metrics.csv", index=False)
    display(shift_table)

    plot_confusion(
        shift_metrics["cm_color_unseen"],
        COLORS,
        "SHIFT/OOD Color Confusion",
        f"{cfg.fig_dir}/cm_color_SHIFT_OOD.png"
    )

    plot_confusion(
        shift_metrics["cm_illum_unseen"],
        ILLUMS,
        "SHIFT/OOD Illumination Confusion",
        f"{cfg.fig_dir}/cm_illum_SHIFT_OOD.png"
    )

    run_policy_analysis(shift_log_path)


# ============================================================
# 14. Reviewer ablations
# ============================================================

ablation_rows = []

if cfg.RUN_FAST_ABLATIONS:
    print("\n================ FAST ABLATIONS ================")

    dl_unseen = make_loader(df_shift_test_unseen, valid_tfms, shuffle=False)

    # Color-only
    color_model = train_color_only(
        df_shift_tr,
        df_shift_val_seen,
        tag="SHIFT_color_only"
    )

    color_metrics = eval_color_only(
        color_model,
        dl_unseen,
        save_log_path=f"{cfg.res_dir}/color_only_shift_prediction_log.csv",
        protocol="SHIFT_OOD_COLOR_ONLY"
    )

    pd.DataFrame([{
        "protocol": "SHIFT_OOD",
        "model": "Color-only",
        "color_macro_f1_unseen": color_metrics["color_macro_f1"],
        "color_accuracy_unseen": color_metrics["color_accuracy"],
        "color_ece_unseen": color_metrics["ece"]
    }]).to_csv(f"{cfg.res_dir}/color_only_shift_metrics.csv", index=False)

    ablation_rows.append({
        "protocol": "SHIFT_OOD",
        "model": "Color-only",
        "color_macro_f1": color_metrics["color_macro_f1"],
        "illum_macro_f1": np.nan,
        "bucket_macro_f1": np.nan,
        "ece": color_metrics["ece"],
        "notes": "Single color head"
    })

    # Multi-task without uncertainty
    mt_no_unc_model, mt_no_unc_val = train_multitask(
        df_shift_tr,
        df_shift_val_seen,
        tag="SHIFT_multitask_no_uncertainty",
        uncertainty=False
    )

    mt_no_unc_log = f"{cfg.res_dir}/multitask_no_uncertainty_shift_prediction_log.csv"

    mt_no_unc_metrics = eval_multitask(
        mt_no_unc_model,
        dl_unseen,
        save_log_path=mt_no_unc_log,
        protocol="SHIFT_OOD_NO_UNCERTAINTY"
    )

    pd.DataFrame([{
        "protocol": "SHIFT_OOD",
        "model": "MultiTask-no-uncertainty",
        "color_macro_f1_unseen": mt_no_unc_metrics["color_macro_f1"],
        "illum_macro_f1_unseen": mt_no_unc_metrics["illum_macro_f1"],
        "bucket_macro_f1_unseen": mt_no_unc_metrics["bucket_macro_f1"],
        "color_ece_unseen": mt_no_unc_metrics["ece"]
    }]).to_csv(f"{cfg.res_dir}/multitask_no_uncertainty_shift_metrics.csv", index=False)

    ablation_rows.append({
        "protocol": "SHIFT_OOD",
        "model": "MultiTask-no-uncertainty",
        "color_macro_f1": mt_no_unc_metrics["color_macro_f1"],
        "illum_macro_f1": mt_no_unc_metrics["illum_macro_f1"],
        "bucket_macro_f1": mt_no_unc_metrics["bucket_macro_f1"],
        "ece": mt_no_unc_metrics["ece"],
        "notes": "Loss = L_color + L_illum"
    })

    # Bucket classifier
    if cfg.RUN_BUCKET_ABLATION:
        bucket_model = train_bucket_model(
            df_shift_tr,
            df_shift_val_seen,
            tag="SHIFT_bucket_classifier"
        )

        bucket_metrics = eval_bucket_model(bucket_model, dl_unseen)

        pd.DataFrame([{
            "protocol": "SHIFT_OOD",
            "model": "Bucket-classifier",
            "color_macro_f1_unseen": bucket_metrics["color_macro_f1"],
            "illum_macro_f1_unseen": bucket_metrics["illum_macro_f1"],
            "bucket_macro_f1_unseen": bucket_metrics["bucket_macro_f1"]
        }]).to_csv(f"{cfg.res_dir}/bucket_shift_metrics.csv", index=False)

        ablation_rows.append({
            "protocol": "SHIFT_OOD",
            "model": "Bucket-classifier",
            "color_macro_f1": bucket_metrics["color_macro_f1"],
            "illum_macro_f1": bucket_metrics["illum_macro_f1"],
            "bucket_macro_f1": bucket_metrics["bucket_macro_f1"],
            "ece": np.nan,
            "notes": "Single 36-way bucket head"
        })

    # Add main result
    main_shift_path = Path(cfg.res_dir) / "shift_ood_metrics.csv"
    if main_shift_path.exists():
        main_shift = pd.read_csv(main_shift_path)

        ablation_rows.append({
            "protocol": "SHIFT_OOD",
            "model": "MultiTask-uncertainty-main",
            "color_macro_f1": float(main_shift.loc[0, "color_macro_f1_unseen"]),
            "illum_macro_f1": float(main_shift.loc[0, "illum_macro_f1_unseen"]),
            "bucket_macro_f1": float(main_shift.loc[0, "bucket_macro_f1_unseen"]),
            "ece": float(main_shift.loc[0, "color_ece_unseen"]),
            "notes": "Main proposed model"
        })

    ablation_df = pd.DataFrame(ablation_rows)
    ablation_df.to_csv(f"{cfg.res_dir}/ablation_summary_for_manuscript.csv", index=False)

    print("\nAblation summary")
    display(ablation_df)


# ============================================================
# 15. LOIO experiments
# ============================================================

if cfg.RUN_LOIO:
    print("\n================ LOIO EXPERIMENTS ================")

    loio_rows = []

    for holdout in ILLUMS:
        print(f"\n---------- LOIO holdout: {holdout} ----------")

        df_loio_train_all = df[df["illum"] != holdout].reset_index(drop=True)
        df_loio_test = df[df["illum"] == holdout].reset_index(drop=True)

        df_loio_tr, df_loio_val = make_group_stratified_fold(
            df_loio_train_all,
            fold=0,
            n_splits=5
        )

        loio_model, loio_val = train_multitask(
            df_loio_tr,
            df_loio_val,
            tag=f"LOIO_holdout_{holdout}",
            uncertainty=True
        )

        dl_loio_test = make_loader(df_loio_test, valid_tfms, shuffle=False)

        loio_log_path = f"{cfg.res_dir}/loio_{holdout}_prediction_log.csv"

        loio_metrics = eval_multitask(
            loio_model,
            dl_loio_test,
            save_log_path=loio_log_path,
            protocol=f"LOIO_{holdout}"
        )

        loio_rows.append({
            "protocol": "LOIO",
            "held_out_illumination": holdout,
            "test_samples": len(df_loio_test),
            "color_macro_f1": loio_metrics["color_macro_f1"],
            "illum_macro_f1": loio_metrics["illum_macro_f1"],
            "bucket_macro_f1": loio_metrics["bucket_macro_f1"],
            "ece": loio_metrics["ece"]
        })

        pd.DataFrame(loio_rows).to_csv(f"{cfg.res_dir}/loio_results_partial.csv", index=False)

    loio_df = pd.DataFrame(loio_rows)
    loio_df.to_csv(f"{cfg.res_dir}/loio_results.csv", index=False)

    print("\nLOIO results")
    display(loio_df)


# ============================================================
# 16. File status and ZIP output
# ============================================================

paths_to_check = {
    "dist_color": f"{cfg.res_dir}/dist_color.csv",
    "dist_illum": f"{cfg.res_dir}/dist_illum.csv",
    "dist_bucket": f"{cfg.res_dir}/dist_bucket.csv",
    "phash_grouping_stats": f"{cfg.res_dir}/phash_grouping_stats.csv",
    "shift_ood_metrics": f"{cfg.res_dir}/shift_ood_metrics.csv",
    "shift_prediction_log": f"{cfg.res_dir}/shift_prediction_log.csv",
    "policy_table": f"{cfg.res_dir}/policy_table_shift_ood.csv",
    "coverage_risk": f"{cfg.res_dir}/coverage_risk_shift_ood.csv",
    "per_color_failure": f"{cfg.res_dir}/per_color_shift_failure_table.csv",
    "per_illum_failure": f"{cfg.res_dir}/per_illum_shift_failure_table.csv",
    "color_only_shift": f"{cfg.res_dir}/color_only_shift_metrics.csv",
    "multitask_no_uncertainty": f"{cfg.res_dir}/multitask_no_uncertainty_shift_metrics.csv",
    "bucket_shift": f"{cfg.res_dir}/bucket_shift_metrics.csv",
    "ablation_summary": f"{cfg.res_dir}/ablation_summary_for_manuscript.csv",
    "loio_results": f"{cfg.res_dir}/loio_results.csv",
}

file_status = pd.DataFrame([
    {"file_key": k, "path": v, "exists": Path(v).exists()}
    for k, v in paths_to_check.items()
])

file_status.to_csv(f"{cfg.res_dir}/file_status.csv", index=False)

print("\nGenerated file status")
display(file_status)

pack_dir = Path("/kaggle/working/cmc_81633_complete_outputs")
pack_dir.mkdir(parents=True, exist_ok=True)

for f in Path(cfg.res_dir).glob("*"):
    if f.is_file():
        shutil.copy(f, pack_dir / f.name)

fig_pack = pack_dir / "figures"
fig_pack.mkdir(exist_ok=True)

for f in Path(cfg.fig_dir).glob("*"):
    if f.is_file():
        shutil.copy(f, fig_pack / f.name)

zip_path = shutil.make_archive(
    "/kaggle/working/cmc_81633_complete_outputs",
    "zip",
    pack_dir
)

print("\nDONE.")
print("Download this ZIP from Kaggle Output panel:")
print(zip_path)